# Data Preparation — GSE225275 (Gastric Antrum)

Load DGE matrices for human, mouse, and rat antrum samples from GSE225275.
Compute basic QC metrics and save as `.h5ad` for downstream analysis.

**Source**: [GSE225275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE225275)  
**Format**: Tab-separated DGE text files (genes x cells)

## 1. Setup & Imports

In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(6, 4))

from importlib.metadata import version
print(f"Scanpy: {version('scanpy')}  |  AnnData: {version('anndata')}  |  NumPy: {np.__version__}  |  Pandas: {pd.__version__}")

Scanpy: 1.10.3  |  AnnData: 0.10.9  |  NumPy: 1.26.4  |  Pandas: 2.2.3


## 2. Project Configuration

In [2]:
PROJECT_NAME = 'gastric_antrum'
GEO_ID = 'GSE225275'

PROJECT_DIR = Path('..')
RAW_DIR = Path('RAW')
OUT_DIR = PROJECT_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Files to load: (filename, species, mt_prefix)
SAMPLES = [
    ('GSE225275_human_data.txt.gz', 'human', 'MT-'),
    ('GSE225275_mouse_data.txt.gz', 'mouse', 'mt-'),
    ('GSE225275_rat_data.txt.gz',   'rat',   'Mt-'),
]

print(f"Project: {PROJECT_NAME}")
print(f"Dataset: {GEO_ID}")
print(f"Raw dir: {RAW_DIR}")
print(f"Output:  {OUT_DIR}")

Project: gastric_antrum
Dataset: GSE225275
Raw dir: RAW
Output:  ../data/processed


## 3. Load Data

In [3]:
adatas = {}

for filename, species, mt_prefix in SAMPLES:
    filepath = RAW_DIR / filename
    print(f"\nLoading {species} from {filename}...")

    # Read DGE: genes x cells -> transpose to cells x genes
    df = pd.read_csv(filepath, sep='\t', index_col=0).T
    print(f"  Raw shape: {df.shape[0]} cells x {df.shape[1]} genes")

    # Create AnnData with sparse matrix
    adata = ad.AnnData(
        X=csr_matrix(df.values),
        obs=pd.DataFrame(index=df.index),
        var=pd.DataFrame(index=df.columns),
    )

    # Add metadata
    sample_id = f'{GEO_ID}_{species}'
    adata.obs['sample_id'] = sample_id
    adata.obs['dataset'] = GEO_ID
    adata.obs['species'] = species
    adata.obs['region'] = 'antrum'
    adata.obs['condition'] = 'WT'

    # Make obs columns categorical
    for col in ['sample_id', 'dataset', 'species', 'region', 'condition']:
        adata.obs[col] = adata.obs[col].astype('category')

    # Ensure var_names are unique
    adata.var_names_make_unique()

    adatas[species] = adata
    print(f"  AnnData: {adata.shape}")

print(f"\nLoaded {len(adatas)} species.")


Loading human from GSE225275_human_data.txt.gz...


  Raw shape: 3813 cells x 14330 genes


  AnnData: (3813, 14330)

Loading mouse from GSE225275_mouse_data.txt.gz...


  Raw shape: 4127 cells x 15107 genes


  AnnData: (4127, 15107)

Loading rat from GSE225275_rat_data.txt.gz...


  Raw shape: 3722 cells x 15396 genes


  AnnData: (3722, 15396)

Loaded 3 species.


## 4. QC Metrics

In [4]:
MT_PREFIXES = {'human': 'MT-', 'mouse': 'mt-', 'rat': 'Mt-'}

for species, adata in adatas.items():
    mt_prefix = MT_PREFIXES[species]
    adata.var['mt'] = adata.var_names.str.startswith(mt_prefix)
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

    n_mt = adata.var['mt'].sum()
    print(f"{species}: {n_mt} mitochondrial genes (prefix '{mt_prefix}')")

human: 1 mitochondrial genes (prefix 'MT-')


mouse: 1 mitochondrial genes (prefix 'mt-')


rat: 0 mitochondrial genes (prefix 'Mt-')


## 5. Summary

In [5]:
summary = []
for species, adata in adatas.items():
    summary.append({
        'species': species,
        'n_cells': adata.n_obs,
        'n_genes': adata.n_vars,
        'median_genes_per_cell': int(np.median(adata.obs['n_genes_by_counts'])),
        'median_counts_per_cell': int(np.median(adata.obs['total_counts'])),
        'median_pct_mito': round(np.median(adata.obs['pct_counts_mt']), 2),
    })

summary_df = pd.DataFrame(summary)
print(f"{'='*70}")
print(f"GSE225275 Summary")
print(f"{'='*70}")
print(summary_df.to_string(index=False))
print(f"{'='*70}")
print(f"Total cells: {summary_df['n_cells'].sum():,}")

GSE225275 Summary
species  n_cells  n_genes  median_genes_per_cell  median_counts_per_cell  median_pct_mito
  human     3813    14330                   2811                    7155             0.14
  mouse     4127    15107                   3363                   20181             0.05
    rat     3722    15396                   2816                   13326             0.00
Total cells: 11,662


## 6. Save

In [6]:
# Save per-species files
for species, adata in adatas.items():
    out_path = OUT_DIR / f'{GEO_ID}_{species}_raw.h5ad'
    adata.write(out_path)
    print(f"Saved {out_path} ({adata.n_obs} cells x {adata.n_vars} genes)")

# Save combined file (outer join keeps all genes; fill_value=0 for missing)
combined = ad.concat(adatas, label='species_key', join='outer', fill_value=0)
combined.obs_names_make_unique()
combined_path = OUT_DIR / f'{GEO_ID}_combined_raw.h5ad'
combined.write(combined_path)
print(f"\nSaved {combined_path} ({combined.n_obs} cells x {combined.n_vars} genes)")

Saved ../data/processed/GSE225275_human_raw.h5ad (3813 cells x 14330 genes)


Saved ../data/processed/GSE225275_mouse_raw.h5ad (4127 cells x 15107 genes)


Saved ../data/processed/GSE225275_rat_raw.h5ad (3722 cells x 15396 genes)


/Users/zahra/miniconda3/envs/bioinfo/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")



Saved ../data/processed/GSE225275_combined_raw.h5ad (11662 cells x 31303 genes)


In [7]:
# Verify saved files
print("Verification:")
for f in sorted(OUT_DIR.glob(f'{GEO_ID}*.h5ad')):
    test = sc.read_h5ad(f)
    print(f"  {f.name}: {test.shape[0]} cells x {test.shape[1]} genes  OK")

Verification:


  GSE225275_combined_raw.h5ad: 11662 cells x 31303 genes  OK


  GSE225275_human_raw.h5ad: 3813 cells x 14330 genes  OK


  GSE225275_mouse_raw.h5ad: 4127 cells x 15107 genes  OK
  GSE225275_rat_raw.h5ad: 3722 cells x 15396 genes  OK
